In [ ]:
import datetime as dt
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import json
import energyid
import typing
from sklearn.linear_model import LinearRegression

from openenergyid.elia import Region

# Set plt figure size
plt.rcParams["figure.figsize"] = (20, 6)

with open("secrets.json") as f:
    secrets = json.load(f)

In [ ]:
eid_client = energyid.PandasClient(
    client_id=secrets["EnergyID_CLIENTID"], client_secret=secrets["EnergyID_CLIENTSECRET"]
)
eid_client.authenticate(
    username=secrets["EnergyID_USERNAME"], password=secrets["EnergyID_PASSWORD"]
)

In [ ]:
start = dt.date(2020, 7, 1)
end = dt.date(2026, 1, 1)

region = Region.East_Flanders

In [ ]:
record = eid_client.get_record(record_id=secrets["RECORDS"]["JAN"])

# This specific record has data from 2021-11-01
start = dt.date(2023, 10, 1)

# The rated power of the system is 147 kWp

In [ ]:
filename = pathlib.Path(f"data/pv_production_{start.isoformat()}_{end.isoformat()}JAN.parquet.gz")

if filename.exists():
    pv_production = pd.read_parquet(filename)
else:
    pv_production = record.get_data(
        name="energyProduction", start=start.isoformat(), end=end.isoformat(), interval="PT1H"
    )
    pv_production = typing.cast(pd.Series, pv_production)
    pv_production = pv_production.to_frame()
    pv_production.to_parquet(filename, compression="gzip")

In [ ]:
pv_production = pv_production.resample("h").sum()

In [ ]:
pv_production.plot()

# Meteo

In [ ]:
meteo = pd.read_parquet("data/openmeteo_hourly_antwerpen.parquet.gz")

In [ ]:
df = pd.merge(pv_production, meteo, left_index=True, right_index=True, how="left")

In [ ]:
df

In [ ]:
df.truncate(
    before=pd.Timestamp("2024-07-06", tz="Europe/Brussels"),
    after=pd.Timestamp("2024-07-07", tz="Europe/Brussels"),
).plot(subplots=True)

In [ ]:
df.dropna(how="any", inplace=True)

In [ ]:
df

In [ ]:
# Drop where every value is 0
df = df[df["energyProduction"] > 0.01]

In [ ]:
df

In [ ]:
import pvlib

In [ ]:
solpos = pvlib.solarposition.get_solarposition(
    time=df.index.map(lambda x: x + pd.Timedelta(minutes=30)),
    latitude=51.20,
    longitude=4.42,
    altitude=10,
)
solpos.index = solpos.index - pd.Timedelta(minutes=30)

In [ ]:
solpos

In [ ]:
tilts = range(20, 40, 1)
azimuths = range(240, 250, 1)

In [ ]:
from itertools import product

In [ ]:
def do_linreg(
    df: pd.DataFrame, x_name: str = "loadfactor", y_name: str = "power", plot: bool = True
):
    reg = LinearRegression(fit_intercept=False, positive=True).fit(X=df[[x_name]], y=df[y_name])
    coef = reg.coef_[0]
    score = reg.score(X=df[[x_name]], y=df[y_name])
    print(f"Coefficient: {coef:.6f}")
    print(f"R^2 score: {score:.4f}")

    if plot:
        # Scatter plot of load factor vs production
        df.plot.scatter(x=x_name, y=y_name)

        # Plot the regression line
        x = df[x_name]
        y = reg.predict(df[[x_name]])
        plt.plot(x, y, color="red")
        plt.show()

    return coef, score

In [ ]:
results = []

for tilt, azimuth in product(tilts, azimuths):
    irr = pvlib.irradiance.get_total_irradiance(
        surface_tilt=tilt,
        surface_azimuth=azimuth,
        solar_zenith=solpos["apparent_zenith"],
        solar_azimuth=solpos["azimuth"],
        ghi=df["shortwave_radiation"],
        dni=df["direct_normal_irradiance"],
        dhi=df["diffuse_radiation"],
    )

    tmp = pd.merge(
        left=df["energyProduction"], right=irr["poa_direct"], left_index=True, right_index=True
    )

    print(f"Fitting model for tilt={tilt} and azimuth={azimuth}")
    coeff, score = do_linreg(tmp, "poa_direct", "energyProduction", plot=False)
    results.append((tilt, azimuth, coeff, score))

In [ ]:
pd.DataFrame(results, columns=["tilt", "azimuth", "coefficient", "r2_score"])

In [ ]:
irr = pvlib.irradiance.get_total_irradiance(
    surface_tilt=29,
    surface_azimuth=246,
    solar_zenith=solpos["apparent_zenith"],
    solar_azimuth=solpos["azimuth"],
    ghi=df["shortwave_radiation"],
    dni=df["direct_normal_irradiance"],
    dhi=df["diffuse_radiation"],
)

tmp = pd.merge(
    left=df["energyProduction"], right=irr["poa_direct"], left_index=True, right_index=True
)

print(f"Fitting model for tilt={tilt} and azimuth={azimuth}")
coeff, score = do_linreg(tmp, "poa_direct", "energyProduction", plot=True)

In [ ]:
tmp = pd.merge(left=df, right=irr, left_index=True, right_index=True)

In [ ]:
tmp.truncate(
    before=pd.Timestamp("2024-07-06", tz="Europe/Brussels"),
    after=pd.Timestamp("2024-07-07", tz="Europe/Brussels"),
).plot(subplots=True)

In [ ]:
train = tmp.truncate(
    before=pd.Timestamp("2023-10-01", tz="Europe/Berlin"),
    after=pd.Timestamp("2024-10-01", tz="Europe/Berlin"),
)

# train = df.truncate(before=pd.Timestamp("2023-10-01", tz="Europe/Berlin"), after=pd.Timestamp("2024-10-01", tz="Europe/Berlin"))

In [ ]:
train

In [ ]:
do_linreg(train, "poa_direct", "energyProduction", plot=True)

In [ ]:
do_linreg(train.resample("D").sum(), "poa_direct", "energyProduction", plot=True)

In [ ]:
do_linreg(train.resample("MS").sum(), "poa_direct", "energyProduction", plot=True)

In [ ]:
def do_binned_linreg(df: pd.DataFrame, remove_outliers: bool = False):
    # Let's put the load factor in  bins and compute the median power for each bin
    tmp = df.copy()

    tmp["poa_bin"] = pd.cut(tmp["poa_direct"], bins=50)

    train_binned = tmp.groupby("poa_bin", observed=True)["energyProduction"].median().reset_index()
    train_binned["midpoint"] = train_binned["poa_bin"].apply(lambda x: x.mid)

    if remove_outliers:
        for _ in range(100):
            train_binned = train_binned[train_binned["energyProduction"].diff().fillna(0) >= 0]

    do_linreg(train_binned, x_name="midpoint", y_name="energyProduction")

In [ ]:
do_binned_linreg(train)

In [ ]:
eval_2024 = tmp.truncate(
    before=pd.Timestamp("2024-01-01", tz="Europe/Berlin"),
    after=pd.Timestamp("2025-01-01", tz="Europe/Berlin"),
)

do_binned_linreg(eval_2024, remove_outliers=False)

In [ ]:
eval_2025 = tmp.truncate(
    before=pd.Timestamp("2025-01-01", tz="Europe/Berlin"),
    after=pd.Timestamp("2026-01-01", tz="Europe/Berlin"),
)

do_binned_linreg(eval_2025, remove_outliers=False)